In [1]:
import geopandas as gpd
import pandas as pd

In [2]:
gdb_file_path = r"/Users/xiaoliu/Work/Project/I-GUIDE/IGUIDE_Aging_Dam-selected/Transmission_Lines/Transmission_Lines.shp"
layer_name = "Transmission_Lines"  # Replace with the actual layer name

# Read the specific layer into a GeoDataFrame
try:
    gdf = gpd.read_file(gdb_file_path, layer=layer_name)
    print(f"Successfully loaded layer: {layer_name}")
except Exception as e:
    print(f"Error loading GDB layer: {e}")

Successfully loaded layer: Transmission_Lines


In [3]:
gdf.columns

Index(['OBJECTID', 'ID', 'TYPE', 'STATUS', 'NAICS_CODE', 'NAICS_DESC',
       'SOURCE', 'SOURCEDATE', 'VAL_METHOD', 'VAL_DATE', 'OWNER', 'VOLTAGE',
       'VOLT_CLASS', 'INFERRED', 'SUB_1', 'SUB_2', 'Shape__Len', 'geometry'],
      dtype='object')

In [4]:
columns_rename = ['OBJECTID', 'ID', 'TYPE', 'STATUS', 'NAICS_CODE', 'NAICS_DESC',
       'SOURCE', 'SOURCEDATE', 'val_method', 'val_date', 'owner', 'voltage',
       'VOLT_CLASS', 'INFERRED', 'SUB_1', 'SUB_2', 'Shape_Len', 'geom']

In [5]:
gdf.columns = columns_rename

In [6]:
gdf = gdf.set_geometry("geom")

In [8]:
target_crs = "EPSG:4326"
reprojected_gdf = gdf.to_crs(target_crs)

print(f"Original CRS: {gdf.crs}")
print(f"New CRS: {reprojected_gdf.crs}")

Original CRS: EPSG:3857
New CRS: EPSG:4326


In [9]:
import psycopg2


# ----------------------------
# Connect to PostGIS
# ----------------------------
conn = psycopg2.connect(
    dbname="utahdaminundationprofiles_aug9_2025",
    user="admin",
    password="admin",
    host="localhost",
    port=5432
)

In [10]:
from sqlalchemy import create_engine
import geopandas as gpd

# --- Assume these variables are derived from your psycopg2 connection details ---
# You must provide these credentials instead of the psycopg2 connection object itself
DB_USER = "admin"
DB_PASS = "admin"
DB_HOST = "localhost"
DB_PORT = "5432"  # Standard PostgreSQL port
DB_NAME = "utahdaminundationprofiles_aug9_2025"
table_name = "transmission"

# Assuming 'merged_gdf' is your GeoDataFrame

# 1. CONSTRUCT THE POSTGRESQL CONNECTION URL
db_url = f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# 2. CREATE THE SQLALCHEMY ENGINE
# GeoPandas requires a SQLAlchemy Engine object for to_postgis()
engine = create_engine(db_url)

# 3. UPLOAD THE GEODATAFRAME
try:
    reprojected_gdf.to_postgis(
        name=table_name,
        con=engine,          # Use the SQLAlchemy engine
        if_exists='replace', # or 'append'
        index=False 
    )
    print(f"Successfully uploaded GeoDataFrame to PostGIS table: {table_name}")
except Exception as e:
    print(f"Error uploading to PostGIS: {e}")

Successfully uploaded GeoDataFrame to PostGIS table: transmission
